In [2]:
from typing import Annotated

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import MemorySaver
from chat_bots import (
                      ChatValuationAsync,
                      ChatFundamentalistasAsync,
                      ChatSentimentoAsync,
                      ChatAnaliseTecnicaAsync)
from tratando_dados import TratandoDadosIndicadores, TratandoDadosValuation, TratarDadosNoticias, tratando_dados_fundamentalistas
from chat_bots import get_secret_key
from pydantic import SecretStr
from typing import Iterator, List, Any
import os
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from selenium import webdriver
import warnings
import pandas as pd
import asyncio
warnings.filterwarnings('ignore')

load_dotenv()

try:
    api_secret_groq = get_secret_key("GROQ_API_KEY")
except KeyError as exc:
    raise ValueError("API key inválida ou não definida") from exc

api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    api_key=api_key,
    model="meta-llama/llama-4-scout-17b-16e-instruct",
    #model="qwen-2.5-coder-32b",
    temperature=0,
    stop_sequences=None,
)

memory = MemorySaver()

class State(TypedDict):
    messages: Annotated[list, add_messages]
    ticker: str
    method_analysis: list[str]
    dados_input : str
    next: str
    
    
class ModeloAnaliseTecnicaAsync():
    def __init__(
        self,
        query: str,
        ticker: str,
        periodo: str = "18Y",
        intervalo: str = "1mo",
        modelo_llm: str = "meta-llama/llama-4-scout-17b-16e-instruct",
        stream: bool = False,
        api_secret: SecretStr | None = api_secret_groq,
    ) -> None:
        self.query = query
        self.ticker = ticker
        self.periodo = periodo
        self.intervalo = intervalo
        self.modelo_llm = modelo_llm
        self.stream = stream
        self.api_secret = api_secret

    def dados_indicadores_tecnicas(self) -> List[Any]:
        ind = TratandoDadosIndicadores(
            ticker=self.ticker, periodo=self.periodo, intervalo=self.intervalo
        )
        return ind.indicadores_data_loader()

    async def chat_analise_tecnica(self) -> str | Iterator[str]:
        dados_tecnicas = self.dados_indicadores_tecnicas()
        response = await ChatAnaliseTecnicaAsync(
            query=self.query,
            dados=dados_tecnicas,
            api_secret=self.api_secret,
            modelo_llm=self.modelo_llm,
            stream=self.stream,
        )
        if '</think>' in response:
            response = response.split('</think>')[1]
        return response

class ModeloValuationAsync:
    def __init__(
        self,
        ticker: str,
        query: str,
        anos_projecao: int = 5,
        taxa_crescimento_perpetuidade: float = 0.014,
        calculo_necessidade_capital_de_giro: bool = False,
        stream: bool = False,
        modelo_llm: str = "meta-llama/llama-4-scout-17b-16e-instruct",
        api_secret: SecretStr | None = api_secret_groq,
    ) -> None:
        self.query = query
        self.ticker = ticker
        self.anos_projecao = anos_projecao
        self.taxa_crescimento_perpetuidade = taxa_crescimento_perpetuidade
        self.calculo_necessidade_capital_de_giro = calculo_necessidade_capital_de_giro
        self.stream = stream
        self.modelo_llm = modelo_llm
        self.api_secret = api_secret

    def tratando_ticker(self) -> str:
        if ".SA" in self.ticker:
            acao = self.ticker
        else:
            acao = f"{self.ticker}.SA"
        return acao

    def dados_valuation(self) -> tuple[str, str, str, str]:
        dados_valu = TratandoDadosValuation(
            ticker=self.tratando_ticker(),
            anos_projecao=5,
            taxa_crescimento_perpetuidade=self.taxa_crescimento_perpetuidade,
            calculo_necessidade_capital_de_giro=self.calculo_necessidade_capital_de_giro,
        )
        markdow_gordon, markdow_fluxo, markdow_preco, markdow_indicadores = (
            dados_valu.dados_valuation()
        )
        return markdow_gordon, markdow_fluxo, markdow_preco, markdow_indicadores

    async def chat_valuation(self) -> str | Iterator[str]:
        markdow_gordon, markdow_fluxo, markdow_preco, markdow_indicadores = (
            self.dados_valuation()
        )

        response = await ChatValuationAsync(
            query=self.query,
            precos_atual_valuations=markdow_preco,
            indicadores_valuation_fluxo=markdow_indicadores,
            valuation_metodo_gordon=markdow_gordon,
            valuation_fluxo_caixa=markdow_fluxo,
            api_secret=self.api_secret,
            modelo_llm=self.modelo_llm,
            stream=self.stream,
        )
        if '</think>' in response:
            response = response.split('</think>')[1]

        return response
    

class ModeloSentimentoAsync:
    def __init__(
        self,
        ticker: str,
        query: str,
        modelo_llm: str = "meta-llama/llama-4-scout-17b-16e-instruct",
        stream: bool = False,
        api_secret_groq: SecretStr | None = api_secret_groq,
        api_secret_serper: SecretStr | None = None,
    ) -> None:
        self.ticker = ticker
        self.query = query
        self.modelo_llm = modelo_llm
        self.stream = stream
        self.api_secret_groq = api_secret_groq
        self.api_secret_serper = api_secret_serper

    def option(self) -> webdriver.ChromeOptions:
        chrome_options = webdriver.ChromeOptions()
        chrome_options.add_argument("--headless")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("--disable-features=NetworkService")
        chrome_options.add_argument("--window-size=1920x1080")
        chrome_options.add_argument("--disable-features=VizDisplayCompositor")
        return chrome_options

    def dados_sentimento(self) -> str:
        dados_noticias = TratarDadosNoticias(
            acao=self.ticker,
            options=self.option(),
            api_secret_groq=self.api_secret_groq,
            api_secret_serper=self.api_secret_serper,
        )
        dados_new = dados_noticias.clean_chat_html_bs4()
        return dados_new

    async def chat_sentimento(self) -> str | Iterator[str]:
        dados_new = self.dados_sentimento()
        response = await ChatSentimentoAsync(
            query=self.query,
            noticia=dados_new,
            api_secret=self.api_secret_groq,
            modelo_llm=self.modelo_llm,
            stream=self.stream,
        )
        if '</think>' in response:
            response = response.split('</think>')[1]
        return response


class ModeloFundamentosAsync:
    def __init__(
        self,
        ticker: str,
        query: str,
        stream: bool = False,
        modelo_llm: str = "meta-llama/llama-4-scout-17b-16e-instruct",
        dados_inicio: str = "2022-06-01",
        api_secret: SecretStr | None = api_secret_groq,
    ) -> None:
        
        self.query = query
        self.ticker = ticker
        self.stream = stream
        self.modelo_llm = modelo_llm
        self.dados_inicio = dados_inicio
        self.api_secret = api_secret

    async def dados_fundamentalistas(self) -> List[Any]:
        return await tratando_dados_fundamentalistas(self.ticker, self.dados_inicio)

    async def chat_fundamentalistas(self) -> str | Iterator[str]:
        dados_fundamentalistas = await self.dados_fundamentalistas()
        response = await ChatFundamentalistasAsync(
            query=self.query,
            dados=dados_fundamentalistas,
            api_secret=self.api_secret,
            modelo_llm=self.modelo_llm,
            stream=self.stream,
        )
        if '</think>' in response:
                response = response.split('</think>')[1]
        return response

In [3]:
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import MessagesState, END
from langgraph.types import Command
async def process_technical(state: State) -> Command[Literal["supervisor"]]:
    """
    Executa a análise técnica para o ticker especificado no estado.

    Recupera o ticker e a última mensagem do usuário do estado.
    Instancia e chama o modelo de análise técnica assíncrono.
    Adiciona a resposta do modelo à lista 'dados_input' no estado.

    Args:
        state (State): O dicionário de estado atual, contendo 'ticker',
                       'messages' e 'dados_input'.

    Returns:
        Optional[State]: O estado atualizado com a resposta da análise técnica
                         adicionada a 'dados_input', ou None se ocorrer um erro.
    """
    print('Entrei nos techincal')
   
    ticker = state['ticker']
    query = state["messages"][-1]
    modelo = ModeloAnaliseTecnicaAsync(ticker=ticker, query=query)
    response = await modelo.chat_analise_tecnica()
    
    dados_input = state['dados_input']
    
    response = dados_input.join(response)
    
    return Command(
        update={"dados_input": [
                HumanMessage(content=response, name="tecnica")
            ]
        },
        goto="supervisor",
    )
    
    

async def process_valuation(state: State) -> Command[Literal["supervisor"]]:
    """
    Executa a análise de valuation para o ticker especificado no estado.

    Recupera o ticker e a última mensagem do usuário do estado.
    Instancia e chama o modelo de valuation assíncrono.
    Adiciona a resposta do modelo à lista 'dados_input' no estado.

    Args:
        state (State): O dicionário de estado atual, contendo 'ticker',
                       'messages' e 'dados_input'.

    Returns:
        Optional[State]: O estado atualizado com a resposta da análise de valuation
                         adicionada a 'dados_input', ou None se ocorrer um erro.
    """
    print('Entrei nos valuation')
    
    ticker = state['ticker']
    query = state["messages"][-1]
    modelo = ModeloValuationAsync(ticker=ticker, query=query)
    response = await modelo.chat_valuation()
    dados_input = state['dados_input']
    response = dados_input.join(response)
    return Command(
        update={"dados_input": [
                HumanMessage(content=response, name="valuation")
            ]
        },
        goto="supervisor",
        )
    
        
    
    
async def process_sentimetal(state: State) -> Command[Literal["supervisor"]]:
    """
    Executa a análise de sentimento para o ticker especificado no estado.

    Recupera o ticker e a última mensagem do usuário do estado.
    Instancia e chama o modelo de análise de sentimento assíncrono.
    Adiciona a resposta do modelo à lista 'dados_input' no estado.

    Args:
        state (State): O dicionário de estado atual, contendo 'ticker',
                       'messages' e 'dados_input'.

    Returns:
        Optional[State]: O estado atualizado com a resposta da análise de sentimento
                         adicionada a 'dados_input', ou None se ocorrer um erro.
    """
    print('Entrei nos sentimental')
    
    ticker = state['ticker']
    query = state["messages"][-1]
    modelo = ModeloSentimentoAsync(ticker=ticker, query=query)
    response= await modelo.chat_sentimento()
    dados_input = state['dados_input']
    response = dados_input.join(response)
    return Command(
        update={"dados_input": [
                HumanMessage(content=response, name="valuation")
            ]
        },
        goto="supervisor",
        )
        
    
async def process_fundamental(state: State) -> Command[Literal["supervisor"]]:
    """
    Executa a análise fundamentalista para o ticker especificado no estado.

    Recupera o ticker e a última mensagem do usuário do estado.
    Instancia e chama o modelo de análise fundamentalista assíncrono.
    Adiciona a resposta do modelo à lista 'dados_input' no estado.

    Args:
        state (State): O dicionário de estado atual, contendo 'ticker',
                       'messages' e 'dados_input'.

    Returns:
        Optional[State]: O estado atualizado com a resposta da análise fundamentalista
                         adicionada a 'dados_input', ou None se ocorrer um erro.
    """
    print('Entrei nos fundamentos')
    
    ticker = state['ticker']
    query = state["messages"][-1]
    modelo = ModeloFundamentosAsync(ticker=ticker, query=query)
    response = await modelo.chat_fundamentalistas()
    dados_input = state['dados_input']
    response = dados_input.join(response)
    return Command(
        update={"dados_input": [
                HumanMessage(content=response, name="valuation")
            ]
        },
        goto="supervisor",
        )
    

async def analise_investimento(state: State) -> Command[Literal["supervisor"]]:
    print('entrei no analis investimento')
    ticker = state['ticker']

    tecnica = ModeloAnaliseTecnicaAsync(
            query=f"Qual a analise tecnica da {ticker}",
            ticker=ticker,
        )
    fundamentos = ModeloFundamentosAsync(
            query=f"Qual o fundamentos da {ticker}",
            ticker=ticker,
        
        )
    valuation = ModeloValuationAsync(
            query=f"Qual o valuation da {ticker}",
            ticker=ticker,
        )
    sentimento = ModeloSentimentoAsync(
            query=f"Qual o sentimento da {ticker}",
            ticker=ticker,
        )
     
    response = await asyncio.gather(
            tecnica.chat_analise_tecnica(),
            fundamentos.chat_fundamentalistas(),
            valuation.chat_valuation(),
            sentimento.chat_sentimento()
        )
        
    dados_input = state['dados_input']
    
    response = dados_input.join(response)
    return Command(
        update={"dados_input": [
                HumanMessage(content=response, name="valuation")
            ]
        },
        goto="supervisor",
        )

In [4]:
async def chat_input(state: State) -> State:
    """
    Processa a última mensagem do usuário para identificar um ticker de ação e o método de análise solicitado.

    Esta função lê uma lista de empresas e seus tickers de uma fonte externa.
    Constrói um prompt para um modelo de linguagem grande (LLM) instruindo-o a extrair
    o ticker da ação e o tipo de análise (fundamentals/valuation/sentimental/technical/Analise_investimento/chatbot)
    da última mensagem do usuário no estado fornecido.

    A função envia o prompt e a mensagem do usuário para o LLM, analisa a resposta
    para extrair o ticker e o método. Realiza uma verificação básica para garantir
    que o ticker extraído seja válido ou encontra um semelhante se necessário.
    Finalmente, atualiza o dicionário de estado com o ticker identificado e o método
    de análise e retorna o estado atualizado.

    Args:
        state (State): Um dicionário representando o estado atual da conversa,
                       espera-se que contenha uma chave "messages" com uma lista
                       de mensagens, onde a última é a do usuário.

    Returns:
        State: O dicionário de estado atualizado com as chaves 'ticker' e
               'method_analysis' preenchidas com os valores extraídos da
               resposta do LLM. O ticker pode ser uma string vazia se nenhum
               for identificado. O método de análise será um dos seguintes:
               fundamentals/valuation/sentimental/technical/Analise_investimento/chatbot.
    """
    
    print('Entrei na chat imput')
    
    empresas_df = pd.read_csv('https://raw.githubusercontent.com/Jeferson100/fundamentalist-stock-brazil/main/dados/setor.csv', encoding='utf-8')
    
    prompt = """
    Você é um assistente especializado em investimentos do mercado brasileiro. Extraia precisamente as seguintes informações:
    
    1. O ticker (código) da ação mencionada na mensagem do usuário. Se houver múltiplos tickers, identifique qual está sendo solicitado para análise.
    2. O método de análise desejado (fundamentals, valuation, sentimental, technical). 
    3. Se for dado exemplos parecido com "Analise essa petrobras" ou "Vale apena investir na vale" responda (Analise_investimento).
    4. Se o usuário apenas mencionar uma empresa sem pedir análise específica, ainda extraia o ticker e defina o método como .
    5. Se não houver menção a nenhuma empresa ou ticker, retorne o método como "chatbot".
    
    Regras para identificação de tickers:
    - Tickers brasileiros geralmente terminam com números (ex: PETR4, VALE3)
    - Se o usuário mencionar apenas o nome da empresa (ex: "Petrobras"), identifique o ticker correto ({empresas_tickers})
    - Se houver ambiguidade, prefira o ticker mais líquido (geralmente terminados em 3 ou 4)
    - Se o usuário mencionar apenas o setor (ex: "bancos"), não retorne ticker específico
    
    Formato de resposta (apenas):
    <FORMATO RESPOSTAS>
    TICKER: [ticker em letras maiúsculas ou vazio se não identificado]
    MÉTODO: [fundamentals/valuation/sentimental/technical/Analise_investimento/chatbot]
    <FORMATO RESPOSTAS>
    """
    
    empresas_tickers = "\n".join([f"{empresa} ({ticker})" for empresa, ticker in 
                                zip(empresas_df['Empresa'], empresas_df['tic'])])
    prompt = prompt.format(empresas_tickers=empresas_tickers)
    
    # Pegar a última mensagem do usuário
    last_message = state["messages"][-1]
    
    # Fazer a chamada ao LLM
    response = await llm.ainvoke([
        HumanMessage(content=prompt),
        HumanMessage(content=last_message.content)
    ])
    
    response_text = response.content
    
    ticker = ''
    metodo_analise = 'chatbot'
    
    # Extração mais robusta
    for line in response_text.split('\n'):
        if line.startswith('TICKER:'):
            ticker = line.split(':')[1].strip()
        elif line.startswith('MÉTODO:'):
            metodo_analise = line.split(':')[1].strip()
    
    if ticker:
        if ticker not in empresas_df['tic'].values:
            similar_tickers = empresas_df[empresas_df['tic'].str.contains(ticker[:2])]['tic'].tolist()
            if similar_tickers:
                ticker = similar_tickers[0]
                
    print(f"Ticker extraído: {ticker}")
    print(f"Método de análise: {metodo_analise}")
    
    state['ticker'] = ticker
    state['method_analysis'] = [metodo_analise]
    
    return state

In [5]:
async def chatbot(state: State) -> dict:
    """
    Gera uma resposta consolidada de análise de investimentos usando um LLM.

    Esta função constrói um prompt detalhado para um LLM (ChatGroq),
    fornecendo-lhe os resultados das análises anteriores (técnica, fundamentalista,
    sentimento, valuation) armazenados em `state['dados_input']` e a última
    mensagem do usuário de `state['messages']`.

    O LLM é instruído a agir como um assistente de investimentos, resumir
    cada tipo de análise, fornecer um comentário geral sobre o ativo,
    indicar um nível de confiança e considerações finais (incluindo riscos),
    formatando a saída em Markdown.

    Args:
        state (State): O dicionário de estado atual, que deve conter:
            - 'dados_input': Uma lista de strings, cada uma representando
                             o resultado de uma análise prévia, separadas
                             por marcadores específicos (ex: '###Technical Analysis###').
            - 'messages': Uma lista de mensagens da conversa, onde a última
                          é a consulta mais recente do usuário.
            - 'ticker': (Opcional) O ticker da ação sendo analisada.
            - 'method_analysis': (Opcional) O método de análise identificado.

    Returns:
        Dict[str, Any]: Um dicionário contendo a nova mensagem da IA ('messages'),
                        e preservando 'ticker', 'method_analysis', e 'dados_input'
                        do estado original. Em caso de erro durante o processamento,
                        a mensagem da IA conterá uma descrição do erro.
    """
    try:
        print('Entrei no chat bot')
        # Criar o prompt do sistema
        system_message = SystemMessage(content=f"""
       Você é um assistente especializado em análise de investimentos no mercado brasileiro.
        
        Voce recebe os seguintes dados 
        <DADOS INPUT>
        {state['dados_input']}.
        <DADOS INPUT>
        Nesses dados voce recebe informacao de analise técnica, analise fundamentalista, analise de sentimento e analise de valuation de acoes expecificado pelos usuarios.
        As analises nesses dados estao divididos pelo seguinte caracteres:
        - #########Technical Analysis#######  = Dentro dessa analise tem analise tecnica da acao.
        - ##########Fundamental Analysis######## = Dentro dessa analise tem a analise dos balancos da acao.
        - ############Valuation Analysis############ = Dentro dessa te a estimativa do valuation da acao.
        - #########Sentiment Analysis##### =  Dentro dessa temos a analise de noticias sobre a acao.
        
        Diretrizes:
        - Seja conciso e direto nas respostas
        - Use linguagem acessível, mas profissional
        - Quando não tiver certeza, admita as limitações
        - Sempre mencione os riscos envolvidos em investimentos
        - Evite recomendações diretas de compra/venda
        
        Voce deve dar o nivel de confianca da analise.
        Indique se analisando as entradas do agentes, a recomendacao e de compra, venda ou manter a acao.
    
        
        """)
        
        # Pegar a última mensagem do usuário
        last_message = state["messages"][-1]
        
        
        # Montar a lista de mensagens para o LLM
        messages = [
            system_message,
            HumanMessage(content=f"{last_message.content}")
        ]
        
        llm = ChatGroq(
        api_key=api_key,
        #model="meta-llama/llama-4-scout-17b-16e-instruct",
        model="qwen-qwq-32b",
        temperature=0,
        stop_sequences=None,
    )
        
        # Fazer a chamada ao LLM
        response = await llm.ainvoke(messages)
        
        # Retornar o estado atualizado
        return {
            "messages": [AIMessage(content=response.content)],
            "ticker": state.get("ticker", ""),
            "method_analysis": state.get("method_analysis", ""),
            "dados_input": state.get('dados_input', "")
        }
        
    except Exception as e:
        return {
            "messages": [AIMessage(content=f"Erro no processamento: {str(e)}")],
            "ticker": state.get("ticker", ""),
            "method_analysis": state.get("method_analysis", ""),
            "dados_input" : ""
 
        }

In [6]:
def should_continue(state: State):
    """Determina o próximo passo com base nos métodos selecionados no estado.

    Verifica se a chave 'selected_methods' no dicionário `state` possui um
    valor considerado verdadeiro (truthy). Se for falso (ou a chave não
    existir e `get` retornar None), a função retorna a string "chatbot".

    Se 'selected_methods' for verdadeiro, a função assume que a chave
    'method_analysis' existe no `state` e contém uma lista não vazia.
    Ela recupera o primeiro elemento dessa lista, atualiza a lista em
    `state` removendo esse primeiro elemento (efeito colateral) e, por fim,
    retorna o elemento que foi recuperado.

    Args:
        state: Um dicionário representando o estado atual. Espera-se que
               contenha potencialmente as chaves:
               - 'selected_methods': Um valor cuja veracidade indica se
                 métodos foram selecionados.
               - 'method_analysis': Uma lista contendo os próximos métodos
                 a serem processados. Esta lista é modificada pela função.

    Returns:
        A string "chatbot" se 'selected_methods' for falso, ou o primeiro
        elemento da lista 'method_analysis' (cujo tipo depende do conteúdo
        da lista) se 'selected_methods' for verdadeiro.

    Raises:
        KeyError: Se 'selected_methods' for verdadeiro, mas a chave
                  'method_analysis' não existir no dicionário `state`.
        IndexError: Se 'selected_methods' for verdadeiro e 'method_analysis'
                   existir, mas for uma lista vazia.

    Side Effects:
        Modifica o dicionário `state` de entrada, removendo o primeiro
        elemento da lista associada à chave 'method_analysis' quando
        'selected_methods' é verdadeiro e a lista não está vazia.
    """
   
    if not state.get("selected_methods"):
        return "chatbot"
    
    method = state["method_analysis"][0]
    
    remaining_methods = state["method_analysis"][1:]
    
    state["method_analysis"] = remaining_methods
    
    return method

In [7]:
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import MessagesState, END
from langgraph.types import Command

# Defina os métodos de análise disponíveis
members = ["fundamentals", "technical", "sentimental", "valuation", "analise_investimento"]
options = members + ["FINISH"]

system_prompt = (
    "Você é um supervisor de análise de investimentos. Sua tarefa é gerenciar "
    f"os seguintes métodos de análise: {members}. Dada a solicitação do usuário, "
    "decida qual próximo método de análise deve ser executado. Cada método "
    "realizará sua análise específica e reportará resultados. Quando todas "
    "as análises necessárias estiverem concluídas, responda com FINISH."
)

class Router(TypedDict):
    """Determina o próximo método de análise. Se nenhum método for necessário, encerra."""
    next: Literal["fundamentals", "technical", "sentimental", "valuation", "analise_investimento", "FINISH"]


llm = ChatGroq(
               api_key=api_key,
    model="qwen-qwq-32b",
    temperature=0,
    stop_sequences=None,
)
        

def supervisor_node(state: State) -> Command[Literal["fundamentals", "technical", "sentimental", "valuation", "analise_investimento", "chatbot"]]:
    print('Entrei no supervisor')
    
    # Se não houver mais métodos de análise, vá para o chatbot
    if not state['method_analysis']:
        print("Todos os métodos de análise concluídos")
        return Command(
            goto="chatbot"
        )

    messages = [
        {"role": "system", "content": system_prompt},
    ] + state['method_analysis']

    # Adiciona contexto sobre os métodos disponíveis
    context_message = (
        f"Métodos de análise disponíveis: {', '.join(state['method_analysis'])}. "
        "Escolha o próximo método a ser executado."
    )
    messages.append({"role": "system", "content": context_message})


    try:
        response = llm.with_structured_output(Router).invoke(messages)
        goto = response["next"]
    except Exception as e:
        print(f"Erro na seleção do método: {e}")
        # Caso de erro, use o primeiro método disponível
        goto = state['method_analysis'][0]
    
    # Se o método selecionado não está na lista de métodos restantes, 
    # escolha o primeiro disponível
    if goto not in state['method_analysis']:
        goto = state['method_analysis'][0]

    # Remover o método selecionado da lista de métodos
    remaining_methods = [m for m in state['method_analysis'] if m != goto]
    
    print(f"Próximo método: {goto}")
    
    return Command(
        goto=goto,
        update={
    
            "method_analysis": remaining_methods,
            "next": goto
        }
    )


In [8]:
def verificacao_tickets(state: State) -> Command[Literal["chat_input","supervisor"]]:
    ticker = state['ticker']
    empresas_df = pd.read_csv('https://raw.githubusercontent.com/Jeferson100/fundamentalist-stock-brazil/main/dados/setor.csv', encoding='utf-8')
    ticks_ok = [tick for tick in ticker if tick in empresas_df['tic'].values]
    if len(ticks_ok) > 0:
        return Command(
            goto="supervisor",
        )
    return Command(
        goto="chat_input",
         )

In [9]:
memory = MemorySaver()
graph_builder = StateGraph(State)
graph_builder.add_node("chat_input", chat_input)
graph_builder.add_node("verificacao_tickets", verificacao_tickets)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("analise_investimento", analise_investimento)
graph_builder.add_node("fundamentals", process_fundamental)
graph_builder.add_node("sentimental", process_sentimetal)
graph_builder.add_node("valuation", process_valuation)
graph_builder.add_node("technical", process_technical)
graph_builder.add_node("supervisor", supervisor_node)
graph_builder.add_edge("chat_input", "verificacao_tickets")
graph_builder.set_entry_point('chat_input')
graph_builder.add_edge('chat_input', 'supervisor')
graph_builder.add_edge("chatbot", END)

graph = graph_builder.compile(memory)

In [ ]:
from IPython.display import Image, display
import nest_asyncio

nest_asyncio.apply()

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass